# Pricing Calculator (Python Module)

## 1 基础参数

In [24]:
import csv
import re
from dataclasses import dataclass, field
from pathlib import Path
from typing import Dict, Iterator, List, Optional


# hardware info
@dataclass
class HardWareInfo:
    device_per_node: int
    device_memory: int
    cost_per_month_per_node: float

    @property
    def cost_per_month_per_device(self) -> float:
        if self.device_per_node <= 0:
            raise ValueError("device_per_node must be > 0")
        return self.cost_per_month_per_node / self.device_per_node


@dataclass
class AscendNPU_910B(HardWareInfo):
    device_per_node: int = 8
    device_memory: int = 64
    cost_per_month_per_node: float = 15_000.0  # CNY/month/node


# deployment info
@dataclass
class DeploymentInfo:
    num_devices: int
    max_model_length: int
    performance: List[dict] = field(default_factory=list)
    name: str = "unnamed_deployment"
    model_name: str = "unknown_model"
    hardware_spec: str = "unknown_hardware"

    def iter_performance(self) -> Iterator[dict]:
        return iter(self.performance)

    def infer_context_tier(self, input_len: int) -> str:
        if input_len < 8_000:
            return "short_context_lt_8k"
        if input_len <= 32_000:
            return "medium_context_8k_to_32k"
        return "long_context_32k_to_64k"


# example
@dataclass
class GLM5_2_on_910B_2D8E(DeploymentInfo):
    num_devices: int = 2 * 8
    max_model_length: int = 40_000
    performance: List[dict] = field(
        default_factory=lambda: [
            {
                "input_len": 1024,
                "output_len": 1024,
                "conc": 1,
                "cache_rate": 0,
                "latency(s)": {
                    "avg": 28.15,
                    "p50": 27.54,
                    "p99": 32.11,
                },
                "ttft(ms)": {
                    "avg": 466.8,
                    "p50": 477.8,
                    "p99": 493.9,
                },
                "tpot(ms)": {
                    "avg": 27.1,
                    "p50": 26.5,
                    "p99": 30.9,
                },
                "throughput(tok/s)": {
                    "total prompt": 37.15,
                    "new_prompt": 37.15,
                    "cached prompt": 0,
                    "completion": 36.37,
                },
            }
        ]
    )
    name: str = "GLM5_2_on_910B_2D8E"
    model_name: str = "GLM-5.2"
    hardware_spec: str = "910B3 x 8 x 2"


# sla
@dataclass(frozen=True)
class SLAItem:
    level: str
    success_rate: str
    ttft_p50: str
    ttft_p90: str
    http_error_rate: str
    tail_behavior: str
    typical_judgment: str


@dataclass
class SLA:
    LEVEL_ORDER: Dict[str, int] = field(default_factory=lambda: {"A": 0, "B": 1, "C": 2, "D": 3, "E": 4})
    tiers: Dict[str, List[SLAItem]] = field(
        default_factory=lambda: {
            "short_context_lt_8k": [
                SLAItem("A", ">=99.9%", "<=1.0s", "<=2.5s", "<=0.1%", "稳定", "更接近原厂或高质量专属接入"),
                SLAItem("B", ">=99.0%", "<=1.5s", "<=4.0s", "<=0.5%", "可接受", "可能为原厂代理或稳定混合路由"),
                SLAItem("C", ">=97.0%", "<=2.5s", "<=6.0s", "<=1.0%", "有波动", "存在明显混合路由或资源竞争"),
                SLAItem("D", ">=95.0%", "<=4.0s", "<=10.0s", "<=3.0%", "长尾明显", "存在导池或不稳定中转风险"),
                SLAItem("E", "<95.0%", ">4.0s", ">10.0s", ">3.0%", "极不稳定", "高风险，不建议生产接入"),
            ],
            "medium_context_8k_to_32k": [
                SLAItem("A", ">=99.5%", "<=1.8s", "<=4.0s", "<=0.2%", "稳定", "中上文处理能力较强，接近高质量原厂或稳定专属接入"),
                SLAItem("B", ">=98.5%", "<=2.5s", "<=6.0s", "<=0.8%", "可接受", "可能为稳定代理或混合路由"),
                SLAItem("C", ">=96.0%", "<=4.0s", "<=9.0s", "<=1.5%", "有波动", "存在一定资源竞争或路由漂移"),
                SLAItem("D", ">=93.0%", "<=6.0s", "<=15.0s", "<=4.0%", "长尾明显", "存在导池或较高不稳定风险"),
                SLAItem("E", "<93.0%", ">6.0s", ">15.0s", ">4.0%", "极不稳定", "高风险，不建议生产接入"),
            ],
            "long_context_32k_to_64k": [
                SLAItem("A", ">=99.0%", "<=3.0s", "<=6.0s", "<=0.5%", "稳定", "长上下文能力与稳定性较强，接近原厂高规格服务"),
                SLAItem("B", ">=97.5%", "<=4.5s", "<=9.0s", "<=1.0%", "可接受", "可能为稳定混合路由或代理接入"),
                SLAItem("C", ">=95.0%", "<=6.5s", "<=14.0s", "<=2.0%", "有波动", "存在明显资源竞争或长文本处理压力"),
                SLAItem("D", ">=90.0%", "<=10.0s", "<=24.0s", "<=5.0%", "长尾明显", "存在导池或不稳定中转风险"),
                SLAItem("E", "<90.0%", ">10.0s", ">24.0s", ">5.0%", "极不稳定", "高风险，不建议生产接入"),
            ],
        }
    )

    def get_tier(self, tier_name: str) -> List[SLAItem]:
        return self.tiers[tier_name]

    def as_dict(self) -> Dict[str, List[dict]]:
        return {
            tier: [
                {
                    "level": item.level,
                    "success_rate": item.success_rate,
                    "ttft_p50": item.ttft_p50,
                    "ttft_p90": item.ttft_p90,
                    "http_error_rate": item.http_error_rate,
                    "tail_behavior": item.tail_behavior,
                    "typical_judgment": item.typical_judgment,
                }
                for item in items
            ]
            for tier, items in self.tiers.items()
        }

    @staticmethod
    def _match_threshold(value: float, expr: str) -> bool:
        if not expr:
            return True
        text = str(expr).strip().replace(" ", "").replace("s", "").replace("%", "")
        for op in ("<=", ">=", "<", ">"):
            if text.startswith(op):
                num_text = text[len(op):]
                try:
                    num = float(num_text)
                except ValueError:
                    return False

                if op == "<=":
                    return value <= num
                if op == "<":
                    return value < num
                if op == ">=":
                    return value >= num
                return value > num
        return False

    def classify_performance(self, deployment: DeploymentInfo) -> List[Dict[str, object]]:
        results: List[Dict[str, object]] = []

        for perf in deployment.iter_performance():
            input_len = int(perf.get("input_len", 0))
            output_len = int(perf.get("output_len", 0))
            conc = perf.get("conc")
            tier = deployment.infer_context_tier(input_len)

            ttft_p50_s = float(perf.get("ttft(ms)", {}).get("p50", 0.0)) / 1000.0
            ttft_p90_proxy_s = float(perf.get("ttft(ms)", {}).get("p99", 0.0)) / 1000.0

            matched_level = "Unrated"
            matched_rule = "No TTFT rule matched"

            for item in self.get_tier(tier):
                ok_p50 = self._match_threshold(ttft_p50_s, item.ttft_p50)
                ok_p90 = self._match_threshold(ttft_p90_proxy_s, item.ttft_p90)
                if ok_p50 and ok_p90:
                    matched_level = item.level
                    matched_rule = f"p50 {ttft_p50_s:.3f}s vs {item.ttft_p50}; p90* {ttft_p90_proxy_s:.3f}s vs {item.ttft_p90}"
                    break

            results.append(
                {
                    "concurrency": conc,
                    "input_len": input_len,
                    "output_len": output_len,
                    "tier": tier,
                    "ttft_p50_s": ttft_p50_s,
                    "ttft_p90_proxy_s": ttft_p90_proxy_s,
                    "sla_level": matched_level,
                    "sla_rule": matched_rule,
                }
            )

        return results


def _to_int_range_max(value: str) -> Optional[int]:
    text = str(value).strip()
    if not text:
        return None
    nums = [int(x) for x in re.findall(r"\d+", text)]
    if not nums:
        return None
    return max(nums)


def _to_float(value: str) -> Optional[float]:
    text = str(value).strip()
    if text == "":
        return None
    try:
        return float(text)
    except ValueError:
        return None


def _num_devices_from_hardware(hardware_text: str) -> int:
    nums = [int(x) for x in re.findall(r"\d+", str(hardware_text))]
    if not nums:
        return 8
    if len(nums) >= 2:
        return nums[-1] * nums[-2]
    return nums[-1]


def load_deployments_from_evalscope_csv(csv_path: str) -> List[DeploymentInfo]:
    csv_file = Path(csv_path)
    if not csv_file.exists():
        raise FileNotFoundError(f"CSV not found: {csv_path}")

    deployments: List[DeploymentInfo] = []
    current_hardware = ""
    current_model = ""
    current_variant = "default"
    perf_rows: List[dict] = []

    def flush_current() -> None:
        nonlocal perf_rows, current_hardware, current_model, current_variant
        if not perf_rows:
            return
        max_input_len = max(int(p["input_len"]) for p in perf_rows)
        deployment_name = f"{current_model}_{current_variant}"
        deployments.append(
            DeploymentInfo(
                name=deployment_name,
                model_name=current_model or "unknown_model",
                hardware_spec=current_hardware or "unknown_hardware",
                num_devices=_num_devices_from_hardware(current_hardware),
                max_model_length=max_input_len,
                performance=perf_rows.copy(),
            )
        )
        perf_rows = []

    with csv_file.open("r", encoding="utf-8-sig", newline="") as f:
        reader = csv.reader(f)
        for row in reader:
            if len(row) < 25:
                row = row + [""] * (25 - len(row))

            hardware = row[0].strip()
            model_name = row[1].strip()
            variant_hint = row[4].strip()

            if hardware:
                if perf_rows:
                    flush_current()
                current_hardware = hardware
                current_model = model_name or current_model
                current_variant = "default"

            if variant_hint and ("tp=" in variant_hint or "dp=" in variant_hint or "2p2d" in variant_hint):
                if perf_rows:
                    flush_current()
                current_variant = variant_hint.replace(",", "_").replace(" ", "")

            input_len = _to_int_range_max(row[8])
            output_len = _to_int_range_max(row[9])
            conc = _to_int_range_max(row[10])
            if input_len is None or output_len is None or conc is None:
                continue

            latency_avg = _to_float(row[12])
            latency_p50 = _to_float(row[13])
            latency_p99 = _to_float(row[14])
            ttft_avg = _to_float(row[15])
            ttft_p50 = _to_float(row[16])
            ttft_p99 = _to_float(row[17])
            tpot_avg = _to_float(row[18])
            tpot_p50 = _to_float(row[19])
            tpot_p99 = _to_float(row[20])
            total_prompt = _to_float(row[21])
            new_prompt = _to_float(row[22])
            cached_prompt = _to_float(row[23])
            completion = _to_float(row[24])
            if completion is None:
                continue

            perf_rows.append(
                {
                    "input_len": int(input_len),
                    "output_len": int(output_len),
                    "conc": int(conc),
                    "cache_rate": 0,
                    "latency(s)": {
                        "avg": float(latency_avg or 0.0),
                        "p50": float(latency_p50 or 0.0),
                        "p99": float(latency_p99 or 0.0),
                    },
                    "ttft(ms)": {
                        "avg": float(ttft_avg or 0.0),
                        "p50": float(ttft_p50 or 0.0),
                        "p99": float(ttft_p99 or 0.0),
                    },
                    "tpot(ms)": {
                        "avg": float(tpot_avg or 0.0),
                        "p50": float(tpot_p50 or 0.0),
                        "p99": float(tpot_p99 or 0.0),
                    },
                    "throughput(tok/s)": {
                        "total prompt": float(total_prompt or 0.0),
                        "new_prompt": float(new_prompt or 0.0),
                        "cached prompt": float(cached_prompt or 0.0),
                        "completion": float(completion),
                    },
                }
            )

    flush_current()
    return deployments


sla = SLA()
print("SLA tiers loaded:", list(sla.tiers.keys()))

SLA tiers loaded: ['short_context_lt_8k', 'medium_context_8k_to_32k', 'long_context_32k_to_64k']


## 2估价模型

In [25]:
from math import ceil, isfinite
from typing import Any, Dict, List, Optional, Tuple


class ProfitModel:
    def __init__(
        self,
        margin: float = 0.55,
        fee_total: float = 0.05,  # payment_fee + bad_debt + tax
        alpha: float = 0.25,
        beta: float = 1.8,
        weight_cache: float = 0.7,
        weight_input: float = 0.2,
        weight_output: float = 0.1,
        premium: float = 1.1 * 1.05 * 1.05,  # K_lat * K_rel * K_reserve
        billing_days_per_month: int = 30,
    ) -> None:
        self.margin = margin
        self.fee_total = fee_total
        self.alpha = alpha
        self.beta = beta
        self.weight_cache = weight_cache
        self.weight_input = weight_input
        self.weight_output = weight_output
        self.premium = premium
        self.billing_days_per_month = billing_days_per_month

    @staticmethod
    def _normalize_market_price_per_token(
        per_token: Optional[float],
        per_million_tokens: Optional[float],
        field_name: str,
    ) -> Optional[float]:
        if per_token is not None:
            return float(per_token)
        if per_million_tokens is None:
            return None
        value = float(per_million_tokens)
        if value < 0:
            raise ValueError(f"{field_name} must be >= 0")
        return value / 1_000_000.0

    def _normalized_mix_weights(self) -> Tuple[float, float, float]:
        w_sum = max(self.weight_cache, 0.0) + max(self.weight_input, 0.0) + max(self.weight_output, 0.0)
        if w_sum <= 0:
            raise ValueError("weight_cache + weight_input + weight_output must be > 0")
        w_cache = max(self.weight_cache, 0.0) / w_sum
        w_input = max(self.weight_input, 0.0) / w_sum
        w_output = max(self.weight_output, 0.0) / w_sum
        return w_cache, w_input, w_output

    @staticmethod
    def _seconds_per_1m_from_tok_s(tok_s: float) -> float:
        return (1_000_000.0 / tok_s) if tok_s > 0 else float("inf")

    def get_pricing(
        self,
        hardware: HardWareInfo,
        deployment: DeploymentInfo,
        sla: Optional[SLA] = None,
        target_sla_level: str = "B",
        manual_sellable_ratio: Optional[float] = None,
        market_input_price_per_token: Optional[float] = None,
        market_output_price_per_token: Optional[float] = None,
        market_cache_price_per_token: Optional[float] = None,
        market_input_price_per_million_tokens: Optional[float] = None,
        market_output_price_per_million_tokens: Optional[float] = None,
        market_cache_price_per_million_tokens: Optional[float] = None,
        cache_fallback_ratio_from_new_prompt: float = 1.0,
    ) -> Dict[str, Any]:
        sla = sla or SLA()

        if cache_fallback_ratio_from_new_prompt < 0:
            raise ValueError("cache_fallback_ratio_from_new_prompt must be >= 0")

        nodes = ceil(deployment.num_devices / hardware.device_per_node)
        monthly_cost = nodes * hardware.cost_per_month_per_node

        net_ratio = 1 - self.margin - self.fee_total
        if net_ratio <= 0:
            raise ValueError("margin + fee_total must be < 1")

        w_cache, w_input, w_output = self._normalized_mix_weights()
        seconds_per_month = self.billing_days_per_month * 24 * 3600

        market_input_price_per_token = self._normalize_market_price_per_token(
            market_input_price_per_token,
            market_input_price_per_million_tokens,
            "market_input_price",
        )
        market_output_price_per_token = self._normalize_market_price_per_token(
            market_output_price_per_token,
            market_output_price_per_million_tokens,
            "market_output_price",
        )
        market_cache_price_per_token = self._normalize_market_price_per_token(
            market_cache_price_per_token,
            market_cache_price_per_million_tokens,
            "market_cache_price",
        )

        has_any_market_price = any(
            x is not None
            for x in (
                market_input_price_per_token,
                market_output_price_per_token,
                market_cache_price_per_token,
            )
        )
        if has_any_market_price and (
            market_input_price_per_token is None
            or market_output_price_per_token is None
            or market_cache_price_per_token is None
        ):
            raise ValueError(
                "If one market price is provided, input/output/cache prices must all be provided"
            )

        level_results = sla.classify_performance(deployment=deployment)
        level_index = {
            (r["input_len"], r["output_len"], r["concurrency"]): r
            for r in level_results
        }

        target_rank = sla.LEVEL_ORDER.get(target_sla_level.upper())
        if target_rank is None:
            raise ValueError(f"Unknown target_sla_level: {target_sla_level}")

        rows: List[Dict[str, Any]] = []

        for perf in deployment.iter_performance():
            input_len = int(perf.get("input_len", 0))
            output_len = int(perf.get("output_len", 0))
            conc = perf.get("conc")

            level_row = level_index.get((input_len, output_len, conc), {})
            sla_level = str(level_row.get("sla_level", "Unrated"))
            sla_rule = str(level_row.get("sla_rule", "No rule"))
            tier = str(level_row.get("tier", deployment.infer_context_tier(input_len)))

            level_rank = sla.LEVEL_ORDER.get(sla_level.upper(), 999)
            sla_pass = level_rank <= target_rank

            throughput = perf.get("throughput(tok/s)", {})
            raw_cache_tok_s = float(throughput.get("cached prompt", 0.0) or 0.0)
            input_tok_s = float(throughput.get("new_prompt", 0.0) or 0.0)
            output_tok_s = float(throughput.get("completion", 0.0) or 0.0)

            if raw_cache_tok_s > 0:
                cache_tok_s = raw_cache_tok_s
                cache_perf_source = "cached_prompt"
                cache_perf_missing = False
                cache_perf_note = ""
            elif input_tok_s > 0:
                cache_tok_s = input_tok_s * float(cache_fallback_ratio_from_new_prompt)
                cache_perf_source = "fallback_new_prompt_ratio"
                cache_perf_missing = True
                cache_perf_note = (
                    "cache throughput missing; used new_prompt throughput * "
                    f"fallback_ratio ({cache_fallback_ratio_from_new_prompt:.4f})"
                )
            else:
                cache_tok_s = 0.0
                cache_perf_source = "missing"
                cache_perf_missing = True
                cache_perf_note = "cache throughput missing and new_prompt throughput unavailable"

            sec_per_1m_cache = self._seconds_per_1m_from_tok_s(cache_tok_s)
            sec_per_1m_input = self._seconds_per_1m_from_tok_s(input_tok_s)
            sec_per_1m_output = self._seconds_per_1m_from_tok_s(output_tok_s)

            blended_sec_per_1m = (
                w_cache * sec_per_1m_cache
                + w_input * sec_per_1m_input
                + w_output * sec_per_1m_output
            )
            blended_tok_s = (
                1_000_000.0 / blended_sec_per_1m
                if blended_sec_per_1m > 0 and isfinite(blended_sec_per_1m)
                else 0.0
            )

            sellable_ratio = (
                (1.0 if sla_pass else 0.0)
                if manual_sellable_ratio is None
                else min(1.0, max(0.0, float(manual_sellable_ratio)))
            )

            effective_tok_s = blended_tok_s * sellable_ratio
            tokens_per_month = seconds_per_month * effective_tok_s

            cached_input_tokens_per_month = tokens_per_month * w_cache
            new_input_tokens_per_month = tokens_per_month * w_input
            output_tokens_per_month = tokens_per_month * w_output

            weighted_token_coef = (
                w_input
                + self.alpha * w_cache
                + self.beta * w_output
            )

            if tokens_per_month > 0 and weighted_token_coef > 0:
                break_even_input_price_per_token = monthly_cost / (tokens_per_month * weighted_token_coef)
                break_even_cache_price_per_token = self.alpha * break_even_input_price_per_token
                break_even_output_price_per_token = self.beta * break_even_input_price_per_token
                break_even_blended_price_per_token = monthly_cost / tokens_per_month

                if has_any_market_price:
                    pricing_mode = "market_forward"
                    applied_input_price_per_token = float(market_input_price_per_token)
                    applied_output_price_per_token = float(market_output_price_per_token)
                    applied_cache_price_per_token = float(market_cache_price_per_token)
                    revenue_per_month = (
                        new_input_tokens_per_month * applied_input_price_per_token
                        + output_tokens_per_month * applied_output_price_per_token
                        + cached_input_tokens_per_month * applied_cache_price_per_token
                    )
                    profit_per_month = revenue_per_month - monthly_cost
                else:
                    pricing_mode = "break_even"
                    applied_input_price_per_token = break_even_input_price_per_token
                    applied_output_price_per_token = break_even_output_price_per_token
                    applied_cache_price_per_token = break_even_cache_price_per_token
                    revenue_per_month = monthly_cost
                    profit_per_month = 0.0

                profit_margin = (profit_per_month / revenue_per_month) if revenue_per_month > 0 else float("nan")

                # Compatibility fields for existing summary/export logic.
                cost_per_1m = break_even_blended_price_per_token * 1_000_000
                price_per_1m = cost_per_1m / net_ratio
                final_per_1m = (
                    (
                        applied_cache_price_per_token * w_cache
                        + applied_input_price_per_token * w_input
                        + applied_output_price_per_token * w_output
                    )
                    * 1_000_000
                )
            else:
                pricing_mode = "no_capacity"
                break_even_input_price_per_token = float("inf")
                break_even_output_price_per_token = float("inf")
                break_even_cache_price_per_token = float("inf")
                break_even_blended_price_per_token = float("inf")

                applied_input_price_per_token = float("inf")
                applied_output_price_per_token = float("inf")
                applied_cache_price_per_token = float("inf")

                revenue_per_month = 0.0
                profit_per_month = -monthly_cost
                profit_margin = float("nan")

                cost_per_1m = float("inf")
                price_per_1m = float("inf")
                final_per_1m = float("inf")

            rows.append(
                {
                    "input_len": input_len,
                    "output_len": output_len,
                    "concurrency": conc,
                    "tier": tier,
                    "sla_level": sla_level,
                    "sla_rule": sla_rule,
                    "target_sla_level": target_sla_level,
                    "sla_pass": sla_pass,
                    "pricing_mode": pricing_mode,
                    "mix_weight_cache": w_cache,
                    "mix_weight_input": w_input,
                    "mix_weight_output": w_output,
                    "blended_cost_per_token": break_even_blended_price_per_token,
                    "revenue_per_month": revenue_per_month,
                    "profit_per_month": profit_per_month,
                    "profit_margin": profit_margin,
                    "cost_per_1m": cost_per_1m,
                    "price_per_1m": price_per_1m,
                    "final_per_1m": final_per_1m,
                    "cache_price_per_1m": applied_cache_price_per_token * 1_000_000,
                    "input_price_per_1m": applied_input_price_per_token * 1_000_000,
                    "output_price_per_1m": applied_output_price_per_token * 1_000_000,
                    "cache_fallback_ratio_from_new_prompt": float(cache_fallback_ratio_from_new_prompt),
                    "cache_perf_source": cache_perf_source,
                    "cache_perf_missing": cache_perf_missing,
                    "cache_perf_note": cache_perf_note,
                    "cache_tok_s_raw": raw_cache_tok_s,
                    "cache_tok_s": cache_tok_s,
                    "input_tok_s": input_tok_s,
                    "output_tok_s": output_tok_s,
                    "sec_per_1m_cache": sec_per_1m_cache,
                    "sec_per_1m_input": sec_per_1m_input,
                    "sec_per_1m_output": sec_per_1m_output,
                    "blended_sec_per_1m": blended_sec_per_1m,
                    "blended_tok_s": blended_tok_s,
                    "goodput_req_per_sec": effective_tok_s,
                    "req_per_sec": effective_tok_s,
                    "req_per_month": tokens_per_month / max(input_len + output_len, 1),
                    "new_input_tokens_per_month": new_input_tokens_per_month,
                    "cached_input_tokens_per_month": cached_input_tokens_per_month,
                    "output_tokens_per_month": output_tokens_per_month,
                    "tokens_per_month": tokens_per_month,
                    "monthly_cost": monthly_cost,
                    "break_even_input_price_per_token": break_even_input_price_per_token,
                    "break_even_output_price_per_token": break_even_output_price_per_token,
                    "break_even_cache_price_per_token": break_even_cache_price_per_token,
                    "applied_input_price_per_token": applied_input_price_per_token,
                    "applied_output_price_per_token": applied_output_price_per_token,
                    "applied_cache_price_per_token": applied_cache_price_per_token,
                    "break_even_input_price_per_million_tokens": break_even_input_price_per_token * 1_000_000,
                    "break_even_output_price_per_million_tokens": break_even_output_price_per_token * 1_000_000,
                    "break_even_cache_price_per_million_tokens": break_even_cache_price_per_token * 1_000_000,
                    "applied_input_price_per_million_tokens": applied_input_price_per_token * 1_000_000,
                    "applied_output_price_per_million_tokens": applied_output_price_per_token * 1_000_000,
                    "applied_cache_price_per_million_tokens": applied_cache_price_per_token * 1_000_000,
                    

                }
            )

        best = None
        for row in rows:
            if not row["sla_pass"] or row["cost_per_1m"] == float("inf"):
                continue
            if best is None or row["cost_per_1m"] < best["cost_per_1m"] or (
                row["cost_per_1m"] == best["cost_per_1m"] and row["blended_tok_s"] > best["blended_tok_s"]
            ):
                best = row

        return {
            "monthly_cost_only_rent": monthly_cost,
            "nodes": nodes,
            "num_devices": deployment.num_devices,
            "rows": rows,
            "best": best,
        }


## 3定义生成流程

In [26]:
import re
from pathlib import Path

try:
    import pandas as pd
except ModuleNotFoundError:
    pd = None

def parse_num_devices_from_spec(spec: str) -> int:
    x_nums = [int(x) for x in re.findall(r"x\s*(\d+)", str(spec), flags=re.IGNORECASE)]
    if not x_nums:
        return 8
    total = 1
    for n in x_nums:
        total *= n
    return total

def _format_summary_display(df, note: str = "SLA not match"):
    if pd is None or df is None:
        return df
    formatted = df.copy().astype(object)
    best_columns = [col for col in formatted.columns if col.startswith("best_")]
    for col in best_columns:
        formatted[col] = formatted[col].fillna(note)
    return formatted

def _format_detail_display(df, note: str = "SLA未满足"):
    if pd is None or df is None or "sla_pass" not in df.columns:
        return df
    formatted = df.copy().astype(object)
    failed_mask = formatted["sla_pass"].fillna(False).astype(bool) == False
    if failed_mask.any():
        for col in formatted.columns:
            if col == "sla_pass":
                continue
            formatted.loc[failed_mask & formatted[col].isna(), col] = note
    return formatted

def _add_best_marker(rows: List[dict], best_row: Optional[dict]) -> List[dict]:
    marked_rows: List[dict] = []
    for row in rows:
        row_copy = dict(row)
        row_copy["best_marker"] = "best" if best_row is not None and row is best_row else ("SLA未满足" if not row.get("sla_pass") else "")
        marked_rows.append(row_copy)
    return marked_rows

def run_pricing_for_all_deployments(
    csv_path: str,
    sla: SLA,
    profit_model: ProfitModel,
    target_sla_level: str = "B",
    market_input_price_per_million_tokens: Optional[float] = None,
    market_output_price_per_million_tokens: Optional[float] = None,
    market_cache_price_per_million_tokens: Optional[float] = None,
    cache_fallback_ratio_from_new_prompt: float = 1.0,
    manual_sellable_ratio: Optional[float] = None,
):
    deployments = load_deployments_from_evalscope_csv(csv_path)
    print(f"Loaded deployments: {len(deployments)}")

    name_counter = {}
    for dep in deployments:
        dep.num_devices = parse_num_devices_from_spec(dep.hardware_spec)
        count = name_counter.get(dep.name, 0) + 1
        name_counter[dep.name] = count
        if count > 1:
            dep.name = f"{dep.name}__{count}"

    results_by_deployment = {}
    summary_rows = []

    for dep in deployments:
        hardware = AscendNPU_910B()
        pricing_result = profit_model.get_pricing(
            hardware=hardware,
            deployment=dep,
            sla=sla,
            target_sla_level=target_sla_level,
            manual_sellable_ratio=manual_sellable_ratio,
            market_input_price_per_million_tokens=market_input_price_per_million_tokens,
            market_output_price_per_million_tokens=market_output_price_per_million_tokens,
            market_cache_price_per_million_tokens=market_cache_price_per_million_tokens,
            cache_fallback_ratio_from_new_prompt=cache_fallback_ratio_from_new_prompt,
        )
        results_by_deployment[dep.name] = pricing_result
        best = pricing_result.get("best")

        summary_rows.append(
            {
                "deployment": dep.name,
                "model_name": dep.model_name,
                "hardware_spec": dep.hardware_spec,
                "num_devices": dep.num_devices,
                "perf_rows": len(dep.performance),
                "monthly_cost": pricing_result["monthly_cost_only_rent"],
                "best_pricing_mode": None if best is None else best["pricing_mode"],
                "best_tier": None if best is None else best["tier"],
                "best_sla_level": None if best is None else best["sla_level"],
                "best_cache_perf_source": None if best is None else best["cache_perf_source"],
                "best_cache_perf_missing": None if best is None else best["cache_perf_missing"],
                "best_cache_fallback_ratio": None if best is None else best["cache_fallback_ratio_from_new_prompt"],
                "best_profit_per_month": None if best is None else best["profit_per_month"],
                "best_profit_margin": None if best is None else best["profit_margin"],
                "best_input_price_per_million_tokens": None if best is None else best["applied_input_price_per_million_tokens"],
                "best_output_price_per_million_tokens": None if best is None else best["applied_output_price_per_million_tokens"],
                "best_cache_price_per_million_tokens": None if best is None else best["applied_cache_price_per_million_tokens"],
            }
        )

    if pd is None:
        print("pandas not installed; showing summary as plain rows:")
        for row in summary_rows:
            print(row)
        print("\nAll deployment detail rows:")
        for dep_name, result in results_by_deployment.items():
            print(f"\n=== {dep_name} ===")
            for row in _add_best_marker(result["rows"], result.get("best")):
                print(row)
        return deployments, results_by_deployment, summary_rows

    summary_df = pd.DataFrame(summary_rows).sort_values(
        by=["best_profit_per_month", "best_profit_margin"],
        ascending=[False, False],
        na_position="last",
    )
    display(_format_summary_display(summary_df))

    detail_columns = [
        "best_marker", "pricing_mode", "tier", "input_len", "output_len", "concurrency", "sla_level",
        "sla_pass", "cache_perf_source", "cache_perf_missing", "cache_fallback_ratio_from_new_prompt", "cache_perf_note",
        "blended_cost_per_token", "applied_input_price_per_million_tokens",
        "applied_output_price_per_million_tokens", "applied_cache_price_per_million_tokens",
        "revenue_per_month", "profit_per_month", "profit_margin"
    ]
    for dep_name in summary_df["deployment"].tolist():
        print(f"Detail rows: {dep_name}")
        detail_rows = _add_best_marker(results_by_deployment[dep_name]["rows"], results_by_deployment[dep_name].get("best"))
        detail_df = pd.DataFrame(detail_rows)[detail_columns]
        display(_format_detail_display(detail_df))

    return deployments, results_by_deployment, summary_df

def export_pricing_results_to_excel(
    summary_output,
    results_by_deployment: dict,
    output_path: str = "pricing_results.xlsx",
):
    if pd is None:
        print("Cannot export Excel because pandas is not installed.")
        return None

    if not hasattr(summary_output, "to_excel"):
        summary_output = pd.DataFrame(summary_output)
    summary_output = _format_summary_display(summary_output)

    out_file = Path(output_path)
    out_file.parent.mkdir(parents=True, exist_ok=True)

    try:
        with pd.ExcelWriter(out_file) as writer:
            summary_output.to_excel(writer, sheet_name="summary", index=False)
            for dep_name, result in results_by_deployment.items():
                sheet_name = dep_name[:31] if dep_name else "deployment"
                detail_rows = _add_best_marker(result["rows"], result.get("best"))
                detail_df = _format_detail_display(pd.DataFrame(detail_rows))
                detail_df.to_excel(writer, sheet_name=sheet_name, index=False)
        print(f"Excel exported: {out_file}")
        return str(out_file)
    except ImportError:
        print("Excel export requires openpyxl or xlsxwriter. Install one of them, then rerun this cell.")
        return None


In [27]:
csv_path = "data/昇腾token工厂测试清单-evalscope.csv"
profit_model = ProfitModel(
    margin=0.0,  # 0.55,
    fee_total=0.0,  # 0.05,  # payment_fee + bad_debt + tax
    # 默认报价体系为 input, output, cache三个，以input为基线
    alpha=0.25,     # price_cache = price_input * alpha
    beta=4,         # price_output = price_input * alpha
    weight_cache=0.7,       # !!! 极大程度影响结论, cache hit 占请求token的占比
    weight_input=0.2,      # !!! 极大程度影响结论, input 占请求token的占比
    weight_output=0.1,     # !!! 极大程度影响结论, output 占请求token的占比
    premium=1,  # 1.1 * 1.05 * 1.05,  # K_lat * K_rel * K_reserve 定价溢价基数
    billing_days_per_month=30,
)

# 直接填你拿到的市场定价（单位：元 / 百万tokens）
# 来自你的示例：输入 8、输出 28、缓存命中 2
# market_input_price_per_million_tokens = 8.0
# market_output_price_per_million_tokens = 28.0
# market_cache_price_per_million_tokens = 2.0

# 如果为None, 测算持平成本的定价
market_input_price_per_million_tokens = None
market_output_price_per_million_tokens = None
market_cache_price_per_million_tokens = None

# 当 cached prompt 缺失时，按 new_prompt * 该比例回退估算 cache 吞吐
# 例如 1.2 代表 cache 吞吐按 new_prompt 的 1.2 倍估算
cache_fallback_ratio_from_new_prompt = 5.0 # !!! 极大程度影响结论 

# 手动指定可售卖比例；None 时沿用 SLA 自动判定
# 调整系数控制实际可售占理论峰值的百分比
manual_sellable_ratio = 1.0 # None # 默认 通过为 1.0, 不通过为 0.0 

# 缓存存储（元/百万tokens/小时）在这版模型中暂未计入，限时免费可视为 0
cache_storage_price_per_million_tokens_per_hour = 0.0

target_sla_level = "B"

deployments, results_by_deployment, summary_output = run_pricing_for_all_deployments(
    csv_path=csv_path,
    sla=sla,
    profit_model=profit_model,
    target_sla_level=target_sla_level,
    market_input_price_per_million_tokens=market_input_price_per_million_tokens,
    market_output_price_per_million_tokens=market_output_price_per_million_tokens,
    market_cache_price_per_million_tokens=market_cache_price_per_million_tokens,
    cache_fallback_ratio_from_new_prompt=cache_fallback_ratio_from_new_prompt,
    manual_sellable_ratio=manual_sellable_ratio,
)

excel_path = export_pricing_results_to_excel(
    summary_output=summary_output,
    results_by_deployment=results_by_deployment,
    output_path="data/pricing_results.xlsx",
)


Loaded deployments: 9


,deployment,model_name,hardware_spec,num_devices,perf_rows,monthly_cost,best_pricing_mode,best_tier,best_sla_level,best_cache_perf_source,best_cache_perf_missing,best_cache_fallback_ratio,best_profit_per_month,best_profit_margin,best_input_price_per_million_tokens,best_output_price_per_million_tokens,best_cache_price_per_million_tokens
0,DeepSeek-V4-Flash-w8a8-mtp_default,DeepSeek-V4-Flash-w8a8-mtp,910B3 x 8,8,1,15000.0,break_even,short_context_lt_8k,A,fallback_new_prompt_ratio,True,5.0,0.0,0.0,88.871005,355.484019,22.217751
1,DeepSeek-V4-Flash-w8a8-mtp_tp=8_dp=1,DeepSeek-V4-Flash-w8a8-mtp,910B3 x 8,8,29,15000.0,break_even,long_context_32k_to_64k,B,fallback_new_prompt_ratio,True,5.0,0.0,0.0,25.544362,102.177447,6.38609
2,DeepSeek-V4-Flash-w8a8-mtp_default__2,DeepSeek-V4-Flash-w8a8-mtp,910B3 x 8,8,1,15000.0,break_even,short_context_lt_8k,A,fallback_new_prompt_ratio,True,5.0,0.0,0.0,94.327134,377.308536,23.581784
3,DeepSeek-V4-Flash-w8a8-mtp_tp=2_dp=4,DeepSeek-V4-Flash-w8a8-mtp,910B3 x 8,8,19,15000.0,break_even,medium_context_8k_to_32k,B,fallback_new_prompt_ratio,True,5.0,0.0,0.0,32.532607,130.130426,8.133152
4,GLM-5.2-w8a8_default,GLM-5.2-w8a8,910B3 x 8 x 2,16,1,30000.0,break_even,medium_context_8k_to_32k,A,fallback_new_prompt_ratio,True,5.0,0.0,0.0,117.193738,468.774951,29.298434
7,GLM-5.2-w8a8_default__2,GLM-5.2-w8a8,910B3 x 8 x 4,32,18,60000.0,break_even,medium_context_8k_to_32k,A,fallback_new_prompt_ratio,True,5.0,0.0,0.0,261.453888,1045.815551,65.363472
8,GLM-5.2-w8a8_default__3,GLM-5.2-w8a8,910B3 x 8 x 4,32,17,60000.0,break_even,long_context_32k_to_64k,A,fallback_new_prompt_ratio,True,5.0,0.0,0.0,83.311701,333.246803,20.827925
5,GLM-5.2-w8a8_dp=2ep=8,GLM-5.2-w8a8,910B3 x 8 x 2,16,9,30000.0,SLA not match,SLA not match,SLA not match,SLA not match,SLA not match,SLA not match,SLA not match,SLA not match,SLA not match,SLA not match,SLA not match
6,DeepSeek-V4-Pro-w4a8-mtp_default,DeepSeek-V4-Pro-w4a8-mtp,910B3 x 8 x 4,32,18,60000.0,SLA not match,SLA not match,SLA not match,SLA not match,SLA not match,SLA not match,SLA not match,SLA not match,SLA not match,SLA not match,SLA not match


Detail rows: DeepSeek-V4-Flash-w8a8-mtp_default


,best_marker,pricing_mode,tier,input_len,output_len,concurrency,sla_level,sla_pass,cache_perf_source,cache_perf_missing,cache_fallback_ratio_from_new_prompt,cache_perf_note,blended_cost_per_token,applied_input_price_per_million_tokens,applied_output_price_per_million_tokens,applied_cache_price_per_million_tokens,revenue_per_month,profit_per_month,profit_margin
0,best,break_even,short_context_lt_8k,1024,1024,1,A,True,fallback_new_prompt_ratio,True,5.0,cache throughput missing; used new_prompt thro...,0.000069,88.871005,355.484019,22.217751,15000.0,0.0,0.0


Detail rows: DeepSeek-V4-Flash-w8a8-mtp_tp=8_dp=1


...

Detail rows: GLM-5.2-w8a8_dp=2ep=8


,best_marker,pricing_mode,tier,input_len,output_len,concurrency,sla_level,sla_pass,cache_perf_source,cache_perf_missing,cache_fallback_ratio_from_new_prompt,cache_perf_note,blended_cost_per_token,applied_input_price_per_million_tokens,applied_output_price_per_million_tokens,applied_cache_price_per_million_tokens,revenue_per_month,profit_per_month,profit_margin
0,SLA未满足,break_even,medium_context_8k_to_32k,8192,256,8,E,False,fallback_new_prompt_ratio,True,5.0,cache throughput missing; used new_prompt thro...,0.000031,40.270879,161.083515,10.06772,30000.0,0.0,0.0
1,SLA未满足,break_even,medium_context_8k_to_32k,8192,256,16,E,False,fallback_new_prompt_ratio,True,5.0,cache throughput missing; used new_prompt thro...,0.000085,110.288932,441.155727,27.572233,30000.0,0.0,0.0
2,SLA未满足,break_even,medium_context_8k_to_32k,8192,256,32,E,False,fallback_new_prompt_ratio,True,5.0,cache throughput missing; used new_prompt thro...,0.000062,80.38117,321.524679,20.095292,30000.0,0.0,0.0
3,SLA未满足,break_even,medium_context_8k_to_32k,8192,256,64,E,False,fallback_new_prompt_ratio,True,5.0,cache throughput missing; used new_prompt thro...,0.000074,95.383169,381.532674,23.845792,30000.0,0.0,0.0
4,SLA未满足,break_even,medium_context_8k_to_32k,8192,256,128,E,False,fallback_new_prompt_ratio,True,5.0,cache throughput missing; used new_prompt thro...,0.000084,108.463026,433.852105,27.115757,30000.0,0.0,0.0
5,SLA未满足,break_even,long_context_32k_to_64k,32684,2048,1,E,False,fallback_new_prompt_ratio,True,5.0,cache throughput missing; used new_prompt thro...,0.000147,190.172926,760.691703,47.543231,30000.0,0.0,0.0
6,SLA未满足,break_even,long_context_32k_to_64k,32684,2048,2,Unrated,False,fallback_new_prompt_ratio,True,5.0,cache throughput missing; used new_prompt thro...,0.000063,81.471321,325.885285,20.36783,30000.0,0.0,0.0
7,SLA未满足,break_even,long_context_32k_to_64k,32684,2048,8,E,False,fallback_new_prompt_ratio,True,5.0,cache throughput missing; used new_prompt thro...,0.000068,88.031745,352.126979,22.007936,30000.0,0.0,0.0
8,SLA未满足,break_even,long_context_32k_to_64k,32684,2048,16,E,False,fallback_new_prompt_ratio,True,5.0,cache throughput missing; used new_prompt thro...,0.000063,81.525977,326.103909,20.381494,30000.0,0.0,0.0


Detail rows: DeepSeek-V4-Pro-w4a8-mtp_default


,best_marker,pricing_mode,tier,input_len,output_len,concurrency,sla_level,sla_pass,cache_perf_source,cache_perf_missing,cache_fallback_ratio_from_new_prompt,cache_perf_note,blended_cost_per_token,applied_input_price_per_million_tokens,applied_output_price_per_million_tokens,applied_cache_price_per_million_tokens,revenue_per_month,profit_per_month,profit_margin
0,SLA未满足,break_even,medium_context_8k_to_32k,8192,256,1,C,False,fallback_new_prompt_ratio,True,5.0,cache throughput missing; used new_prompt thro...,0.000147,189.855402,759.421608,47.463851,60000.0,0.0,0.0
1,SLA未满足,break_even,medium_context_8k_to_32k,8192,256,8,Unrated,False,fallback_new_prompt_ratio,True,5.0,cache throughput missing; used new_prompt thro...,0.000085,109.936908,439.747631,27.484227,60000.0,0.0,0.0
2,SLA未满足,break_even,medium_context_8k_to_32k,8192,256,16,E,False,fallback_new_prompt_ratio,True,5.0,cache throughput missing; used new_prompt thro...,0.000077,99.431164,397.724657,24.857791,60000.0,0.0,0.0
3,SLA未满足,break_even,medium_context_8k_to_32k,8192,256,32,E,False,fallback_new_prompt_ratio,True,5.0,cache throughput missing; used new_prompt thro...,0.000057,74.051478,296.20591,18.512869,60000.0,0.0,0.0
4,SLA未满足,break_even,medium_context_8k_to_32k,8192,256,64,E,False,fallback_new_prompt_ratio,True,5.0,cache throughput missing; used new_prompt thro...,0.000049,63.62405,254.496199,15.906012,60000.0,0.0,0.0
5,SLA未满足,break_even,medium_context_8k_to_32k,8192,256,128,E,False,fallback_new_prompt_ratio,True,5.0,cache throughput missing; used new_prompt thro...,0.000055,70.817081,283.268325,17.70427,60000.0,0.0,0.0
6,SLA未满足,break_even,long_context_32k_to_64k,32684,2048,1,D,False,fallback_new_prompt_ratio,True,5.0,cache throughput missing; used new_prompt thro...,0.000157,202.192835,808.771339,50.548209,60000.0,0.0,0.0
7,SLA未满足,break_even,long_context_32k_to_64k,32684,2048,8,E,False,fallback_new_prompt_ratio,True,5.0,cache throughput missing; used new_prompt thro...,0.000042,54.395759,217.583035,13.59894,60000.0,0.0,0.0
8,SLA未满足,break_even,long_context_32k_to_64k,32684,2048,16,E,False,fallback_new_prompt_ratio,True,5.0,cache throughput missing; used new_prompt thro...,0.000028,36.223818,144.895271,9.055954,60000.0,0.0,0.0
9,SLA未满足,break_even,long_context_32k_to_64k,32684,2048,32,E,False,fallback_new_prompt_ratio,True,5.0,cache throughput missing; used new_prompt thro...,0.000018,23.805572,95.222288,5.951393,60000.0,0.0,0.0


Excel exported: data\pricing_results.xlsx
